# Music Store Customer Support Multi-Agent System

A LangGraph-based multi-agent customer support system for a digital music
store, backed by the **Chinook** SQLite database (loaded entirely
in-memory over the network — no external database setup required).

**Architecture at a glance:**

```
START -> verify_customer (HITL) -> load_memory -> supervisor
                                                        |-- music_catalog --\
                                                        |-- invoice_info ----+--> save_memory -> END
                                                        |-- both: music_catalog -> invoice_info -> synthesize -> save_memory -> END
                                                        |-- FINISH -> END
```

This notebook mirrors the `src/` package 1:1, with markdown commentary for
each section, and ends with the two required test cases.

> **Setup reminder:** copy `.env.example` to `.env` and set `GROQ_API_KEY`
> before running. Internet access is required on first run to download the
> Chinook SQL script.


In [ ]:
import sys, os
if os.path.basename(os.getcwd()) != "music_store_support" and "music_store_support" not in sys.path[0]:
    pass  # adjust sys.path.append(...) here if needed
from dotenv import load_dotenv
load_dotenv()
import os
assert os.environ.get("GROQ_API_KEY"), "Set GROQ_API_KEY in your .env file before running this notebook."
print("Environment OK.")

Environment OK.


## 1. Database setup — Chinook (in-memory)

We load the Chinook SQL script directly from GitHub into an in-memory SQLite database. This is the exact helper code specified by the project requirements.

In [2]:
"""Chinook database loader.

Loads the Chinook SQLite database entirely in-memory by downloading the
official SQL creation script and executing it against an in-memory SQLite
connection. This avoids the need for any external database setup.

NOTE: The helper function below is used exactly as specified by the
project requirements and must not have its behavior modified.
"""

from __future__ import annotations

import sqlite3

import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool


def get_engine_for_chinook_db():
    """Download the Chinook SQL script and load it into an in-memory SQLite DB.

    Returns:
        A SQLAlchemy Engine backed by a single shared in-memory SQLite
        connection (via StaticPool) so that all sessions see the same data.
    """
    url = (
        "https://raw.githubusercontent.com/lerocha/chinook-database/"
        "master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    )
    response = requests.get(url)
    sql_script = response.text
    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


# Module-level singletons so every part of the app shares the same
# in-memory database instance.
engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)


def print_table_list() -> list[str]:
    """Print and return the list of tables available in the Chinook DB."""
    tables = db.get_usable_table_names()
    print("Chinook tables available:", sorted(tables))
    return sorted(tables)


# (module-level `engine` / `db` singletons are created on import above)


C:\Users\user\AppData\Local\Temp\ipykernel_31804\97165477.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [3]:
print_table_list()  # confirm Customer, Invoice, InvoiceLine, Artist, Album, Track, Genre, Employee are accessible

Chinook tables available: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

## 2. Graph state schema

Short-term (per-thread) memory: `customer_id`, the full `messages` history, and `loaded_memory` (a summary of long-term preferences injected into agent prompts). We also carry `remaining_steps` (required by `create_react_agent`) and an internal `route` scratch field used to drive conditional edges.

In [4]:
"""Graph state schema shared across all nodes and sub-agents."""

from __future__ import annotations

from typing import Annotated, Optional

from langgraph.graph.message import AnyMessage, add_messages
from langgraph.managed.is_last_step import RemainingSteps
from typing_extensions import NotRequired, TypedDict


class State(TypedDict):
    """Short-term (per-thread) memory carried through the graph.

    Attributes:
        customer_id: The verified Chinook CustomerId for this session.
            Empty string until verification succeeds.
        messages: Full conversation history (auto-merged via add_messages).
        loaded_memory: A short natural-language summary of the customer's
            long-term music preferences, injected into agent prompts.
        remaining_steps: Managed value LangGraph uses to guard against
            infinite agent loops.
        route: Internal scratch field holding the supervisor's most recent
            routing decision ("music_catalog" | "invoice_info" | "both" |
            "FINISH"), used only to drive conditional edges. Not part of the
            three required fields, but needed to make routing explicit and
            testable.
    """

    customer_id: str
    messages: Annotated[list[AnyMessage], add_messages]
    loaded_memory: str
    remaining_steps: RemainingSteps
    route: NotRequired[str]


## 3. Shared LLM configuration

Centralized here so the model can be swapped easily. Uses **Groq** (`ChatGroq`), defaulting to `llama-3.3-70b-versatile` at `temperature=0`, reading `GROQ_API_KEY` from the environment.

In [5]:
"""Shared LLM configuration.

Centralizing model construction here makes it easy to swap providers
without touching every agent module. Currently configured for **Groq**.
"""

from __future__ import annotations

import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

DEFAULT_MODEL_NAME = os.environ.get("MODEL_NAME", "llama-3.3-70b-versatile")


def get_llm(temperature: float = 0.0, model_name: str | None = None) -> ChatGroq:
    """Construct a ChatGroq instance.

    Args:
        temperature: Sampling temperature. Defaults to 0 for deterministic,
            support-oriented responses.
        model_name: Optional override of the default model name. Falls back
            to the MODEL_NAME env var, then "llama-3.3-70b-versatile".

    Returns:
        A configured ChatGroq instance. Reads the API key from the
        GROQ_API_KEY environment variable.
    """
    if not os.environ.get("GROQ_API_KEY"):
        raise EnvironmentError(
            "GROQ_API_KEY is not set. Copy .env.example to .env and set "
            "your key, or export it in your shell before running."
        )
    return ChatGroq(model=model_name or DEFAULT_MODEL_NAME, temperature=temperature)


## 4. Music Catalog Sub-Agent tools

All queries use SQLAlchemy **bound parameters** — user input is never concatenated into SQL text. Each tool also demonstrates the second HITL scenario: if a required argument (artist / genre / song title) is blank, the tool itself calls `interrupt()` to ask the user for it, then resumes with the reply.

In [6]:
"""Tools for the Music Catalog Sub-Agent.

All tools query the Chinook database via SQLAlchemy using bound
parameters only -- user-supplied strings are NEVER concatenated directly
into SQL text.
"""

from __future__ import annotations

from typing import Any

from langchain_core.tools import tool
from langgraph.types import interrupt
from sqlalchemy import text

from src.db import engine


def _require_nonblank(value: str, ask: str) -> str:
    """Ensure a required tool argument is non-blank, interrupting if not.

    If ``value`` is blank (empty/whitespace), this pauses the graph via
    ``interrupt()`` and asks the user for the missing detail. When the
    graph is resumed, the user's reply is used in place of the original
    (missing) value.

    Args:
        value: The argument value supplied by the calling agent/LLM.
        ask: The clarification question to show the user if value is blank.

    Returns:
        The original value if non-blank, otherwise the user's resumed reply.
    """
    if value and value.strip():
        return value.strip()
    reply = interrupt({"question": ask})
    return str(reply).strip()


def _rows_to_dicts(result) -> list[dict[str, Any]]:
    """Convert a SQLAlchemy CursorResult into a list of plain dicts."""
    return [dict(row._mapping) for row in result]


@tool
def get_albums_by_artist(artist: str) -> list[dict[str, Any]]:
    """Return albums released by the given artist.

    Args:
        artist: The artist's name (partial, case-insensitive match allowed).

    Returns:
        A list of dicts with keys: ArtistName, AlbumId, Title.
    """
    artist = _require_nonblank(
        artist, "Which artist would you like albums for?"
    )
    query = text(
        """
        SELECT ar.Name AS ArtistName, al.AlbumId AS AlbumId, al.Title AS Title
        FROM Artist ar
        JOIN Album al ON al.ArtistId = ar.ArtistId
        WHERE ar.Name LIKE :artist
        ORDER BY al.Title
        """
    )
    with engine.connect() as conn:
        result = conn.execute(query, {"artist": f"%{artist}%"})
        return _rows_to_dicts(result)


@tool
def get_tracks_by_artist(artist: str) -> list[dict[str, Any]]:
    """Return tracks recorded by the given artist.

    If no exact/partial match is found for the artist, this falls back to
    a broader search across similarly-named artists.

    Args:
        artist: The artist's name (partial, case-insensitive match allowed).

    Returns:
        A list of dicts with keys: ArtistName, AlbumTitle, TrackName.
    """
    artist = _require_nonblank(
        artist, "Which artist would you like tracks for?"
    )
    query = text(
        """
        SELECT ar.Name AS ArtistName, al.Title AS AlbumTitle, t.Name AS TrackName
        FROM Artist ar
        JOIN Album al ON al.ArtistId = ar.ArtistId
        JOIN Track t ON t.AlbumId = al.AlbumId
        WHERE ar.Name LIKE :artist
        ORDER BY al.Title, t.TrackId
        """
    )
    with engine.connect() as conn:
        result = conn.execute(query, {"artist": f"%{artist}%"})
        rows = _rows_to_dicts(result)

    if rows:
        return rows

    # Fallback: try splitting the artist name into words and matching any of them.
    words = [w for w in artist.split() if len(w) > 2]
    if not words:
        return []

    fallback_query = text(
        """
        SELECT ar.Name AS ArtistName, al.Title AS AlbumTitle, t.Name AS TrackName
        FROM Artist ar
        JOIN Album al ON al.ArtistId = ar.ArtistId
        JOIN Track t ON t.AlbumId = al.AlbumId
        WHERE ar.Name LIKE :word
        ORDER BY al.Title, t.TrackId
        LIMIT 50
        """
    )
    with engine.connect() as conn:
        result = conn.execute(fallback_query, {"word": f"%{words[0]}%"})
        return _rows_to_dicts(result)


@tool
def get_songs_by_genre(genre: str) -> list[dict[str, Any]]:
    """Return songs that belong to the given genre.

    Args:
        genre: The genre name (partial, case-insensitive match allowed),
            e.g. "Rock", "Jazz", "Metal".

    Returns:
        A list of dicts with keys: GenreName, ArtistName, TrackName.
    """
    genre = _require_nonblank(
        genre, "Which genre are you interested in?"
    )
    query = text(
        """
        SELECT g.Name AS GenreName, ar.Name AS ArtistName, t.Name AS TrackName
        FROM Genre g
        JOIN Track t ON t.GenreId = g.GenreId
        JOIN Album al ON al.AlbumId = t.AlbumId
        JOIN Artist ar ON ar.ArtistId = al.ArtistId
        WHERE g.Name LIKE :genre
        ORDER BY ar.Name, t.Name
        LIMIT 100
        """
    )
    with engine.connect() as conn:
        result = conn.execute(query, {"genre": f"%{genre}%"})
        return _rows_to_dicts(result)


@tool
def check_for_songs(song_title: str) -> list[dict[str, Any]]:
    """Check whether a song with the given (partial) title exists.

    Args:
        song_title: The song title, or a fragment of it, to search for.
            Matching is case-insensitive and allows partial matches.

    Returns:
        A list of dicts with keys: TrackName, ArtistName, AlbumTitle.
    """
    song_title = _require_nonblank(
        song_title, "What is the song title (or part of it) you're looking for?"
    )
    query = text(
        """
        SELECT t.Name AS TrackName, ar.Name AS ArtistName, al.Title AS AlbumTitle
        FROM Track t
        JOIN Album al ON al.AlbumId = t.AlbumId
        JOIN Artist ar ON ar.ArtistId = al.ArtistId
        WHERE t.Name LIKE :song_title
        ORDER BY t.Name
        LIMIT 50
        """
    )
    with engine.connect() as conn:
        result = conn.execute(query, {"song_title": f"%{song_title}%"})
        return _rows_to_dicts(result)


MUSIC_TOOLS = [
    get_albums_by_artist,
    get_tracks_by_artist,
    get_songs_by_genre,
    check_for_songs,
]


## 5. Invoice Information Sub-Agent tools

Same bound-parameter discipline. `get_employee_by_invoice_and_customer` demonstrates the same missing-argument interrupt pattern for a missing `invoice_id`.

In [7]:
"""Tools for the Invoice Information Sub-Agent.

All tools query the Chinook database via SQLAlchemy using bound
parameters only -- user-supplied strings are NEVER concatenated directly
into SQL text.
"""

from __future__ import annotations

from typing import Any

from langchain_core.tools import tool
from langgraph.types import interrupt
from sqlalchemy import text

from src.db import engine


def _rows_to_dicts(result) -> list[dict[str, Any]]:
    """Convert a SQLAlchemy CursorResult into a list of plain dicts."""
    return [dict(row._mapping) for row in result]


def _require_nonblank(value: str, ask: str) -> str:
    """Ensure a required tool argument is non-blank, interrupting if not.

    See ``music_tools._require_nonblank`` for the identical pattern used
    here for the Invoice Information Sub-Agent's tools.
    """
    if value and str(value).strip():
        return str(value).strip()
    reply = interrupt({"question": ask})
    return str(reply).strip()


@tool
def get_invoices_by_customer_sorted_by_date(customer_id: str) -> list[dict[str, Any]]:
    """Return all invoices for a customer, sorted by InvoiceDate descending.

    Args:
        customer_id: The Chinook CustomerId (as a string or int-like value).

    Returns:
        A list of dicts with keys: InvoiceId, InvoiceDate, BillingAddress,
        Total.
    """
    query = text(
        """
        SELECT InvoiceId, InvoiceDate, BillingAddress, Total
        FROM Invoice
        WHERE CustomerId = :customer_id
        ORDER BY InvoiceDate DESC
        """
    )
    with engine.connect() as conn:
        result = conn.execute(query, {"customer_id": customer_id})
        return _rows_to_dicts(result)


@tool
def get_invoices_sorted_by_unit_price(customer_id: str) -> list[dict[str, Any]]:
    """Return all invoice line items for a customer, sorted by UnitPrice descending.

    Args:
        customer_id: The Chinook CustomerId (as a string or int-like value).

    Returns:
        A list of dicts with keys: InvoiceId, TrackId, UnitPrice, Quantity.
    """
    query = text(
        """
        SELECT il.InvoiceId AS InvoiceId, il.TrackId AS TrackId,
               il.UnitPrice AS UnitPrice, il.Quantity AS Quantity
        FROM InvoiceLine il
        JOIN Invoice i ON i.InvoiceId = il.InvoiceId
        WHERE i.CustomerId = :customer_id
        ORDER BY il.UnitPrice DESC
        """
    )
    with engine.connect() as conn:
        result = conn.execute(query, {"customer_id": customer_id})
        return _rows_to_dicts(result)


@tool
def get_employee_by_invoice_and_customer(
    invoice_id: str, customer_id: str
) -> dict[str, Any]:
    """Return the support employee associated with a given invoice and customer.

    Traverses Customer -> SupportRepId -> Employee, after confirming the
    invoice belongs to the given customer.

    Args:
        invoice_id: The Chinook InvoiceId.
        customer_id: The Chinook CustomerId.

    Returns:
        A dict with keys: FirstName, LastName, Email, Phone, Title. Empty
        dict if no matching employee/invoice/customer combination exists.
    """
    invoice_id = _require_nonblank(
        invoice_id,
        "Which invoice ID would you like the support contact for?",
    )
    query = text(
        """
        SELECT e.FirstName AS FirstName, e.LastName AS LastName,
               e.Email AS Email, e.Phone AS Phone, e.Title AS Title
        FROM Invoice i
        JOIN Customer c ON c.CustomerId = i.CustomerId
        JOIN Employee e ON e.EmployeeId = c.SupportRepId
        WHERE i.InvoiceId = :invoice_id AND i.CustomerId = :customer_id
        """
    )
    with engine.connect() as conn:
        result = conn.execute(
            query, {"invoice_id": invoice_id, "customer_id": customer_id}
        )
        rows = _rows_to_dicts(result)
        return rows[0] if rows else {}


INVOICE_TOOLS = [
    get_invoices_by_customer_sorted_by_date,
    get_invoices_sorted_by_unit_price,
    get_employee_by_invoice_and_customer,
]


## 6. Music Catalog Sub-Agent

Built with LangGraph's `create_react_agent` prebuilt, bound to the music tools above. The prompt is built dynamically per-turn to inject `state["loaded_memory"]` for personalization.

In [8]:
"""Music Catalog Sub-Agent.

A ReAct-style agent (LangGraph prebuilt) bound to the music catalog tools.
It personalizes recommendations using ``state["loaded_memory"]``.
"""

from __future__ import annotations

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

from src.llm import get_llm
from src.state import State
from src.tools.music_tools import MUSIC_TOOLS

MUSIC_AGENT_SYSTEM_PROMPT = """You are the Music Catalog Sub-Agent for a \
digital music store's customer support system.

Your job is to answer questions about albums, tracks, genres, and to give \
music recommendations, using ONLY the provided tools to look up real data \
from the Chinook catalog. Never invent album or track names.

Rules:
- Always call a tool to look up catalog information before answering.
- If the user's request is missing a required detail (e.g. no artist name, \
no genre, no song title given), call the relevant tool anyway with an \
empty value for that field -- the tool itself will ask the user for the \
missing detail and pause until they answer.
- When you have results, summarize them clearly and concisely for the \
customer (e.g. a bulleted list of album titles or song names).
- If the customer's long-term preferences are provided below, use them to \
personalize recommendations when relevant.

Customer's known music preferences:
{loaded_memory}
"""


def _build_prompt(state: State):
    """Build the message list for the music agent, injecting long-term memory."""
    loaded_memory = state.get("loaded_memory") or "No preferences on file yet."
    system_text = MUSIC_AGENT_SYSTEM_PROMPT.format(loaded_memory=loaded_memory)
    return [SystemMessage(content=system_text)] + state["messages"]


def build_music_agent():
    """Construct the Music Catalog Sub-Agent as a compiled LangGraph agent."""
    llm = get_llm()
    return create_react_agent(
        llm,
        tools=MUSIC_TOOLS,
        prompt=_build_prompt,
        state_schema=State,
    )


music_agent = build_music_agent()


C:\Users\user\AppData\Local\Temp\ipykernel_31804\3077456341.py:49: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


## 7. Invoice Information Sub-Agent

Same pattern, injecting the **verified** `state["customer_id"]` into its system prompt so it never has to ask the customer to repeat their identity.

In [9]:
"""Invoice Information Sub-Agent.

A ReAct-style agent (LangGraph prebuilt) bound to the invoice tools. It has
access to the verified ``customer_id`` from short-term (graph) state, so it
never needs to ask the customer to repeat their identity.
"""

from __future__ import annotations

from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

from src.llm import get_llm
from src.state import State
from src.tools.invoice_tools import INVOICE_TOOLS

INVOICE_AGENT_SYSTEM_PROMPT = """You are the Invoice Information Sub-Agent \
for a digital music store's customer support system.

Your job is to answer questions about a customer's invoices/purchases, \
including totals, dates, most expensive items, and which support employee \
handled a given invoice. Use ONLY the provided tools -- never invent \
numbers or dates.

The verified customer for this session has CustomerId = {customer_id}. \
Always use this CustomerId when calling tools; do not ask the customer to \
repeat it.

Rules:
- Always call a tool to look up real invoice data before answering.
- If a tool needs an invoice ID that hasn't been specified (e.g. asking \
"who was my support rep for this invoice" without saying which one), call \
the tool with an empty value for that field -- the tool will ask the user \
for the missing detail and pause until they answer.
- Present amounts with a currency symbol and dates in a readable format.
- Be concise and factual.
"""


def _build_prompt(state: State):
    """Build the message list for the invoice agent, injecting customer_id."""
    customer_id = state.get("customer_id") or "UNKNOWN"
    system_text = INVOICE_AGENT_SYSTEM_PROMPT.format(customer_id=customer_id)
    return [SystemMessage(content=system_text)] + state["messages"]


def build_invoice_agent():
    """Construct the Invoice Information Sub-Agent as a compiled LangGraph agent."""
    llm = get_llm()
    return create_react_agent(
        llm,
        tools=INVOICE_TOOLS,
        prompt=_build_prompt,
        state_schema=State,
    )


invoice_agent = build_invoice_agent()


C:\Users\user\AppData\Local\Temp\ipykernel_31804\826640537.py:50: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


## 8. Supervisor agent

Uses a Pydantic structured output (`RouteDecision`) to choose `"music_catalog"`, `"invoice_info"`, `"both"`, or `"FINISH"` for the latest message.

In [10]:
"""Supervisor agent: routes customer queries to the correct sub-agent(s)."""

from __future__ import annotations

from typing import Literal

from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field

from src.llm import get_llm
from src.state import State

SUPERVISOR_SYSTEM_PROMPT = """You are the Supervisor for a digital music \
store's customer support system. You do not answer questions yourself -- \
you decide which specialist(s) should handle the customer's latest \
message.

Specialists available:
- "music_catalog": albums, tracks, genres, song lookups, and \
recommendations.
- "invoice_info": invoices, purchase history, prices paid, and which \
support employee handled an invoice.
- "both": the message clearly needs BOTH specialists (e.g. it asks about \
a purchase AND about albums/tracks in the same message).
- "FINISH": the conversation's latest turn has already been fully \
answered by the specialists and nothing further is needed right now.

Customer's known music preferences (for context only, does not affect \
routing): {loaded_memory}

Read the latest customer message and choose exactly one route.
"""


class RouteDecision(BaseModel):
    """Structured routing decision produced by the supervisor."""

    route: Literal["music_catalog", "invoice_info", "both", "FINISH"] = Field(
        description="Which specialist(s) should handle the latest message."
    )
    reasoning: str = Field(
        description="One short sentence explaining the routing choice."
    )


def build_supervisor_router():
    """Build a callable that returns a RouteDecision for the current state."""
    llm = get_llm()
    structured_llm = llm.with_structured_output(RouteDecision)

    def route(state: State) -> RouteDecision:
        loaded_memory = state.get("loaded_memory") or "No preferences on file yet."
        system_text = SUPERVISOR_SYSTEM_PROMPT.format(loaded_memory=loaded_memory)
        messages = [SystemMessage(content=system_text)] + state["messages"]
        try:
            decision = structured_llm.invoke(messages)
        except Exception as exc:  # noqa: BLE001 - surface a safe fallback
            print(f"[supervisor] routing call failed, defaulting to FINISH: {exc}")
            decision = RouteDecision(
                route="FINISH", reasoning="Routing call failed; ending turn safely."
            )
        return decision

    return route


## 9. Human-in-the-loop helpers

Thin wrappers around `langgraph.types.interrupt()` so verification and clarification prompts read consistently.

In [11]:
"""Human-in-the-loop helper utilities.

These helpers wrap ``langgraph.types.interrupt`` so that verification and
clarification prompts are phrased consistently across the graph.
"""

from __future__ import annotations

from typing import Any

from langgraph.types import interrupt


def ask_for_identity() -> str:
    """Pause the graph and ask the user to verify their identity.

    Returns:
        The raw string the user supplies on resume (expected to be a
        Customer ID, phone number, or email address).
    """
    payload: dict[str, Any] = {
        "type": "identity_verification",
        "question": (
            "I couldn't verify your identity. Please provide your "
            "Customer ID, Phone, or Email so I can verify your identity."
        ),
    }
    return str(interrupt(payload)).strip()


def ask_for_clarification(question: str) -> str:
    """Pause the graph and ask the user a free-form clarifying question.

    Args:
        question: The clarification question to display to the user.

    Returns:
        The raw string the user supplies on resume.
    """
    payload: dict[str, Any] = {"type": "clarification", "question": question}
    return str(interrupt(payload)).strip()


## 10. Customer verification node

Parses the latest user message for a Customer ID, email, or phone number (normalizing spaces/parentheses/dashes/leading `+`), resolves it against the Chinook `Customer` table, and **interrupts** — looping until resolved — if nothing matches.

In [12]:
"""Customer verification node.

Parses the latest user message for a customer_id, email, or phone number,
resolves it to a Chinook CustomerId, and stores it in short-term state. If
no valid identity can be resolved, the node interrupts and asks the user to
supply one, looping until verification succeeds.
"""

from __future__ import annotations

import re
from typing import Optional

from sqlalchemy import text

from src.db import engine
from src.nodes.hitl import ask_for_identity
from src.state import State

_EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_CUSTOMER_ID_RE = re.compile(r"\bcustomer\s*id\D{0,5}(\d+)\b", re.IGNORECASE)
_BARE_ID_RE = re.compile(r"^\s*(\d{1,10})\s*$")
_PHONE_RE = re.compile(r"[+()\d][\d()+\-.\s]{5,}\d")


def _normalize_phone(raw: str) -> str:
    """Strip spaces, parentheses, dashes, and a leading '+' from a phone string."""
    digits = re.sub(r"[\s()\-]", "", raw)
    digits = digits.lstrip("+")
    return digits


def _lookup_by_customer_id(customer_id: str) -> Optional[str]:
    query = text("SELECT CustomerId FROM Customer WHERE CustomerId = :cid")
    with engine.connect() as conn:
        row = conn.execute(query, {"cid": customer_id}).fetchone()
        return str(row[0]) if row else None


def _lookup_by_email(email: str) -> Optional[str]:
    query = text("SELECT CustomerId FROM Customer WHERE LOWER(Email) = LOWER(:email)")
    with engine.connect() as conn:
        row = conn.execute(query, {"email": email}).fetchone()
        return str(row[0]) if row else None


def _lookup_by_phone(phone: str) -> Optional[str]:
    """Match a normalized phone number against normalized Customer.Phone values."""
    normalized_input = _normalize_phone(phone)
    if not normalized_input:
        return None
    query = text("SELECT CustomerId, Phone FROM Customer WHERE Phone IS NOT NULL")
    with engine.connect() as conn:
        rows = conn.execute(query).fetchall()
    for customer_id, db_phone in rows:
        if _normalize_phone(db_phone) == normalized_input:
            return str(customer_id)
        # Also allow suffix match in case of differing country-code prefixes.
        if normalized_input.endswith(_normalize_phone(db_phone)[-9:]) and len(
            _normalize_phone(db_phone)
        ) >= 9:
            return str(customer_id)
    return None


def _try_resolve_customer_id(user_text: str) -> Optional[str]:
    """Attempt to resolve a Chinook CustomerId from free-form user text."""
    if not user_text:
        return None

    match = _CUSTOMER_ID_RE.search(user_text)
    if match:
        resolved = _lookup_by_customer_id(match.group(1))
        if resolved:
            return resolved

    bare_match = _BARE_ID_RE.match(user_text)
    if bare_match:
        resolved = _lookup_by_customer_id(bare_match.group(1))
        if resolved:
            return resolved

    email_match = _EMAIL_RE.search(user_text)
    if email_match:
        resolved = _lookup_by_email(email_match.group(0))
        if resolved:
            return resolved

    phone_match = _PHONE_RE.search(user_text)
    if phone_match:
        resolved = _lookup_by_phone(phone_match.group(0))
        if resolved:
            return resolved

    return None


def verify_customer_node(state: State) -> dict:
    """Resolve and verify the customer's identity, interrupting if needed.

    Args:
        state: Current graph state. Expects the latest HumanMessage to
            possibly contain a customer_id, phone, or email.

    Returns:
        A partial state update setting ``customer_id``.
    """
    if state.get("customer_id"):
        return {"customer_id": state["customer_id"]}

    latest_text = ""
    for msg in reversed(state["messages"]):
        role = getattr(msg, "type", None)
        if role == "human":
            latest_text = msg.content if isinstance(msg.content, str) else str(msg.content)
            break

    customer_id = _try_resolve_customer_id(latest_text)
    while not customer_id:
        user_reply = ask_for_identity()
        customer_id = _try_resolve_customer_id(user_reply)

    return {"customer_id": customer_id}


## 11. Long-term memory nodes

`load_memory` reads a per-customer preferences summary from `InMemoryStore` (namespaced by `customer_id`) into `state["loaded_memory"]`. `save_memory` runs a small structured-output LLM extraction over the recent conversation to update that summary (merged + deduplicated).

In [13]:
"""Long-term memory nodes backed by langgraph.store.memory.InMemoryStore.

Preferences are namespaced per customer: ("customer", customer_id,
"preferences"). We store a single key "summary" holding a concise,
deduplicated natural-language description of the customer's known music
preferences (favorite artists/genres).
"""

from __future__ import annotations

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from pydantic import BaseModel, Field

from src.llm import get_llm
from src.state import State

_PREFERENCES_KEY = "summary"


def _namespace(customer_id: str) -> tuple[str, str, str]:
    return ("customer", customer_id, "preferences")


def load_memory_node(state: State, config: RunnableConfig, *, store: BaseStore) -> dict:
    """Load the customer's saved preferences into short-term state.

    Args:
        state: Current graph state (must already contain a verified
            customer_id).
        config: LangGraph runnable config (unused here, required by the
            store-injection calling convention).
        store: The long-term memory store, auto-injected by LangGraph.

    Returns:
        A partial state update setting ``loaded_memory``.
    """
    customer_id = state.get("customer_id", "")
    if not customer_id:
        return {"loaded_memory": "No preferences on file yet."}

    item = store.get(_namespace(customer_id), _PREFERENCES_KEY)
    if item and item.value.get("text"):
        return {"loaded_memory": item.value["text"]}
    return {"loaded_memory": "No preferences on file yet."}


class ExtractedPreferences(BaseModel):
    """Structured extraction of music preferences from a conversation."""

    has_preferences: bool = Field(
        description="Whether the conversation reveals any concrete music "
        "preference (specific artist, genre, or song the customer likes)."
    )
    summary: str = Field(
        default="",
        description="A short (<=25 words), deduplicated, comma-separated "
        "summary of the customer's favorite artists/genres, merging any "
        "prior known preferences with new ones. Empty if has_preferences "
        "is False.",
    )


def save_memory_node(state: State, config: RunnableConfig, *, store: BaseStore) -> dict:
    """Extract and persist music preferences revealed in the conversation.

    Args:
        state: Current graph state.
        config: LangGraph runnable config (unused here).
        store: The long-term memory store, auto-injected by LangGraph.

    Returns:
        An empty partial state update (this node only has the side effect
        of writing to the long-term store).
    """
    customer_id = state.get("customer_id", "")
    if not customer_id:
        return {}

    existing_item = store.get(_namespace(customer_id), _PREFERENCES_KEY)
    existing_text = existing_item.value.get("text", "") if existing_item else ""

    recent_messages = state["messages"][-6:]
    conversation_snippet = "\n".join(
        f"{getattr(m, 'type', 'unknown')}: {m.content}"
        for m in recent_messages
        if isinstance(m.content, str)
    )

    llm = get_llm()
    structured_llm = llm.with_structured_output(ExtractedPreferences)
    extraction_prompt = [
        SystemMessage(
            content=(
                "Extract the customer's music preferences (artists, genres, "
                "songs they like) from the conversation below. Merge with "
                "any previously known preferences, deduplicate, and keep the "
                "summary concise (<=25 words).\n\n"
                f"Previously known preferences: {existing_text or 'none'}"
            )
        ),
        HumanMessage(content=conversation_snippet or "(no content)"),
    ]

    try:
        extracted = structured_llm.invoke(extraction_prompt)
    except Exception as exc:  # noqa: BLE001 - never let memory extraction crash the turn
        print(f"[save_memory] preference extraction failed, skipping save: {exc}")
        return {}

    if extracted.has_preferences and extracted.summary.strip():
        store.put(
            _namespace(customer_id),
            _PREFERENCES_KEY,
            {"text": extracted.summary.strip()},
        )

    return {}


## 12. Graph orchestration

Wires every node above into a single `StateGraph`, compiled with `InMemorySaver` (checkpointer, for short-term memory + HITL resume) and `InMemoryStore` (long-term memory).

In [14]:
"""Builds and compiles the full customer-support StateGraph.

Graph shape (see README.md for the ASCII diagram):

    START -> verify_customer -> load_memory -> supervisor
                                                    |-- music_catalog --\
                                                    |-- invoice_info ----+--> save_memory -> END
                                                    |-- both: music_catalog -> invoice_info -> synthesize -> save_memory -> END
                                                    |-- FINISH -> END
"""

from __future__ import annotations

from langchain_core.messages import AIMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore

from src.agents.invoice_agent import invoice_agent
from src.agents.music_agent import music_agent
from src.agents.supervisor import build_supervisor_router
from src.llm import get_llm
from src.nodes.memory import load_memory_node, save_memory_node
from src.nodes.verify_customer import verify_customer_node
from src.state import State

_route_query = build_supervisor_router()


def supervisor_node(state: State) -> dict:
    """Decide routing for the latest customer message and record it in state."""
    decision = _route_query(state)
    print(f"[supervisor] route={decision.route} ({decision.reasoning})")
    return {"route": decision.route}


def music_catalog_node(state: State) -> dict:
    """Run the Music Catalog Sub-Agent on the current state."""
    result = music_agent.invoke(state)
    return {"messages": result["messages"]}


def invoice_info_node(state: State) -> dict:
    """Run the Invoice Information Sub-Agent on the current state."""
    result = invoice_agent.invoke(state)
    return {"messages": result["messages"]}


def synthesize_node(state: State) -> dict:
    """Combine the music + invoice sub-agent replies into one final answer.

    Only used when the supervisor routed to "both". Looks back over the
    most recent AI messages produced by the two sub-agents and asks the LLM
    to merge them into a single, well-organized customer-facing reply.
    """
    recent_ai_texts = [
        m.content
        for m in state["messages"][-6:]
        if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip()
    ]
    if not recent_ai_texts:
        return {}

    llm = get_llm()
    try:
        synthesis = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Combine the following sub-agent replies into one "
                        "clear, well-organized final answer for the "
                        "customer. Keep all concrete facts (amounts, dates, "
                        "album/track names) exactly as given; do not "
                        "invent anything new."
                    )
                ),
                *[AIMessage(content=t) for t in recent_ai_texts],
            ]
        )
    except Exception as exc:  # noqa: BLE001
        print(f"[synthesize] LLM call failed, skipping synthesis: {exc}")
        return {}

    return {"messages": [synthesis]}


def route_from_supervisor(state: State) -> str:
    """Conditional edge: choose the first destination after the supervisor."""
    route = state.get("route", "FINISH")
    if route in ("music_catalog", "both"):
        return "music_catalog"
    if route == "invoice_info":
        return "invoice_info"
    return "end"


def route_after_music(state: State) -> str:
    """Conditional edge: after music_catalog, continue to invoice_info if 'both'."""
    if state.get("route") == "both":
        return "invoice_info"
    return "save_memory"


def route_after_invoice(state: State) -> str:
    """Conditional edge: after invoice_info, synthesize if this was a 'both' route."""
    if state.get("route") == "both":
        return "synthesize"
    return "save_memory"


def build_graph():
    """Construct and compile the full multi-agent StateGraph.

    Returns:
        A tuple of (compiled_graph, store) so callers can inspect/seed the
        long-term memory store directly if needed (e.g. in tests).
    """
    builder = StateGraph(State)

    builder.add_node("verify_customer", verify_customer_node)
    builder.add_node("load_memory", load_memory_node)
    builder.add_node("supervisor", supervisor_node)
    builder.add_node("music_catalog", music_catalog_node)
    builder.add_node("invoice_info", invoice_info_node)
    builder.add_node("synthesize", synthesize_node)
    builder.add_node("save_memory", save_memory_node)

    builder.add_edge(START, "verify_customer")
    builder.add_edge("verify_customer", "load_memory")
    builder.add_edge("load_memory", "supervisor")

    builder.add_conditional_edges(
        "supervisor",
        route_from_supervisor,
        {"music_catalog": "music_catalog", "invoice_info": "invoice_info", "end": END},
    )
    builder.add_conditional_edges(
        "music_catalog",
        route_after_music,
        {"invoice_info": "invoice_info", "save_memory": "save_memory"},
    )
    builder.add_conditional_edges(
        "invoice_info",
        route_after_invoice,
        {"synthesize": "synthesize", "save_memory": "save_memory"},
    )
    builder.add_edge("synthesize", "save_memory")
    builder.add_edge("save_memory", END)

    checkpointer = InMemorySaver()
    store = InMemoryStore()
    graph = builder.compile(checkpointer=checkpointer, store=store)
    return graph, store


graph, memory_store = build_graph()


In [15]:
# Visualize the compiled graph's node list (Mermaid/PNG rendering requires
# extra optional dependencies; the node list alone confirms the wiring).
print(list(graph.get_graph().nodes))

['__start__', 'verify_customer', 'load_memory', 'supervisor', 'music_catalog', 'invoice_info', 'synthesize', 'save_memory', '__end__']


## 13. Interactive loop & demo runner

`run_conversation(thread_id)` drives a real stdin/stdout chat loop, detecting interrupts and resuming with `Command(resume=...)`. `run_demo()` runs the two required test cases non-interactively.

In [16]:
"""Interactive CLI loop and non-interactive demo runner for the graph."""

from __future__ import annotations

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.types import Command

from src.graph import graph


def _print_ai_messages(result: dict) -> None:
    """Print any new AI messages found in a graph invocation result."""
    for m in result.get("messages", []):
        if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
            print(f"\nAssistant: {m.content}\n")


def _handle_interrupts(result: dict, thread_id: str) -> dict:
    """Print interrupt prompts and resume the graph with stdin input until done.

    Args:
        result: The most recent graph.invoke()/graph result (may contain an
            "__interrupt__" key when the graph has paused).
        thread_id: The thread id used for this conversation.

    Returns:
        The final result dict once no interrupt remains.
    """
    config = {"configurable": {"thread_id": thread_id}}
    while "__interrupt__" in result:
        interrupt_obj = result["__interrupt__"][0]
        question = interrupt_obj.value.get("question", "Please provide more info:")
        print(f"\n[Assistant needs more info] {question}")
        user_reply = input("You: ")
        result = graph.invoke(Command(resume=user_reply), config=config)
    return result


def run_conversation(thread_id: str) -> None:
    """Run an interactive stdin/stdout conversation loop against the graph.

    Args:
        thread_id: A unique identifier for this conversation's checkpoint
            thread. Reusing the same thread_id resumes the same session
            (short-term memory, including customer_id, persists).
    """
    config = {"configurable": {"thread_id": thread_id}}
    print("Music Store Customer Support (type 'quit' to exit)")
    while True:
        user_input = input("\nYou: ")
        if user_input.strip().lower() in {"quit", "exit"}:
            print("Goodbye!")
            break

        result = graph.invoke(
            {"messages": [HumanMessage(content=user_input)]}, config=config
        )
        result = _handle_interrupts(result, thread_id)
        _print_ai_messages(result)


def _invoke_and_resolve(thread_id: str, user_input: str, resume_replies: list[str]) -> dict:
    """Helper for the non-interactive demo: invoke and auto-resume interrupts.

    Args:
        thread_id: Checkpoint thread id.
        user_input: The initial human message text.
        resume_replies: Canned replies to feed to successive interrupts, in
            order, standing in for a real user during the scripted demo.

    Returns:
        The final graph result dict.
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)

    reply_iter = iter(resume_replies)
    while "__interrupt__" in result:
        interrupt_obj = result["__interrupt__"][0]
        question = interrupt_obj.value.get("question", "Please provide more info:")
        print(f"[interrupt] {question}")
        try:
            reply = next(reply_iter)
        except StopIteration:
            print("[demo] No more scripted replies available; stopping.")
            break
        print(f"[demo user reply] {reply}")
        result = graph.invoke(Command(resume=reply), config=config)
    return result


def run_demo() -> None:
    """Run the two required test-case scenarios end-to-end and print results."""
    print("\n=== Test Case 1: verified customer, combined music+invoice query ===")
    result = _invoke_and_resolve(
        thread_id="demo-thread-1",
        user_input=(
            "My phone number is +55 (12) 3923-5555. How much was my most "
            "recent purchase? What albums do you have by the Rolling Stones?"
        ),
        resume_replies=[],
    )
    _print_ai_messages(result)

    print("\n=== Test Case 1 (continued): long-term memory recall ===")
    config = {"configurable": {"thread_id": "demo-thread-1"}}
    result2 = graph.invoke(
        {"messages": [HumanMessage(content="List some songs that match my preferences?")]},
        config=config,
    )
    result2 = _handle_interrupts_demo(result2, "demo-thread-1")
    _print_ai_messages(result2)

    print("\n=== Test Case 2: new session, no credentials -> identity interrupt ===")
    config2 = {"configurable": {"thread_id": "demo-thread-2"}}
    result3 = graph.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "How much was my most recent purchase? What albums "
                        "do you have by the Rolling Stones?"
                    )
                )
            ]
        },
        config=config2,
    )
    if "__interrupt__" in result3:
        question = result3["__interrupt__"][0].value.get("question")
        print(f"[interrupt raised, as expected] {question}")
    else:
        print("[unexpected] No interrupt was raised for an unverified customer.")


def _handle_interrupts_demo(result: dict, thread_id: str) -> dict:
    """Same as _handle_interrupts but never blocks on stdin during the demo."""
    config = {"configurable": {"thread_id": thread_id}}
    while "__interrupt__" in result:
        question = result["__interrupt__"][0].value.get("question")
        print(f"[interrupt] {question} (no scripted reply provided; stopping)")
        break
    return result


if __name__ == "__main__":
    run_demo()



=== Test Case 1: verified customer, combined music+invoice query ===
[supervisor] route=both (The customer asks for the amount of their most recent purchase (invoice info) and also requests a list of Rolling Stones albums (catalog). Both specialists are required.)

Assistant: I’m sorry, but I don’t have access to your purchase history or billing details in this system, so I can’t tell you the amount of your most recent purchase. For that information, please contact our billing department or provide your order number so we can look it up in the appropriate system.

Here are the Rolling Stones albums currently in our catalog:

- **Hot Rocks, 1964‑1971 (Disc 1)**
- **No Security**
- **Voodoo Lounge**

Let me know if you’d like more details about any of these albums (track listings, pricing, etc.) or if there’s anything else I can help you with!


Assistant: Your most recent purchase was **$8.91** (Invoice #382 dated August 7 2025).

Here are the Rolling Stones albums we have in our catal

## 14. Test Case 1 — verified customer, combined music + invoice query

**Turn 1:** *"My phone number is +55 (12) 3923-5555. How much was my most recent purchase? What albums do you have by the Rolling Stones?"*

Expected: the phone resolves to Chinook `CustomerId=1`, the supervisor routes to **both** sub-agents, and the reply includes the most recent purchase amount and a list of Rolling Stones albums.

**Turn 2 (same thread):** *"List some songs that match my preferences?"* — should use the long-term memory saved from Turn 1 (Rolling Stones / Rock) to recommend songs.


In [17]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.types import Command

thread_id_1 = "notebook-test-case-1"
config_1 = {"configurable": {"thread_id": thread_id_1}}

result = graph.invoke(
    {"messages": [HumanMessage(content=(
        "My phone number is +55 (12) 3923-5555. How much was my most recent "
        "purchase? What albums do you have by the Rolling Stones?"
    ))]},
    config=config_1,
)

# Resolve any interrupts automatically (shouldn't be needed here, since the
# phone number matches a real Chinook customer).
while "__interrupt__" in result:
    q = result["__interrupt__"][0].value.get("question")
    print("[interrupt]", q)
    break  # no scripted reply for an unexpected interrupt in this demo

print("Verified customer_id:", result.get("customer_id"))
for m in result["messages"]:
    if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
        print("\nAssistant:", m.content)

[supervisor] route=both (The customer asks for the amount of their most recent purchase (invoice info) and also requests a list of Rolling Stones albums (catalog). Both specialists are required.)
Verified customer_id: 1

Assistant: I’m sorry—I don’t have access to your purchase history or billing details, so I can’t tell you the amount of your most recent order. If you need that information, please contact our billing department or check your order confirmation email.

Here are the Rolling Stones albums currently in our catalog:

- **Hot Rocks, 1964‑1971 (Disc 1)**
- **No Security**
- **Voodoo Lounge**

Let me know if you’d like more details about any of these albums (track listings, pricing, availability, etc.) or if you’d like recommendations based on the Rolling Stones style!

Assistant: Your most recent purchase (Invoice #382 on August 7 2025) was **$8.91**.  

Rolling Stones albums we have:

- Hot Rocks, 1964‑1971 (Disc 1)  
- No Security  
- Voodoo Lounge  

Let me know if you’d 

In [18]:
# Turn 2: preference-based recommendation, using long-term memory saved from Turn 1.
result_2 = graph.invoke(
    {"messages": [HumanMessage(content="List some songs that match my preferences?")]},
    config=config_1,
)
for m in result_2["messages"]:
    if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
        print("Assistant:", m.content)

[supervisor] route=music_catalog (The user is asking for song recommendations that match their known preference (The Rolling Stones), which is a catalog query.)
Assistant: I’m sorry—I don’t have access to your purchase history or billing details, so I can’t tell you the amount of your most recent order. If you need that information, please contact our billing department or check your order confirmation email.

Here are the Rolling Stones albums currently in our catalog:

- **Hot Rocks, 1964‑1971 (Disc 1)**
- **No Security**
- **Voodoo Lounge**

Let me know if you’d like more details about any of these albums (track listings, pricing, availability, etc.) or if you’d like recommendations based on the Rolling Stones style!
Assistant: Your most recent purchase (Invoice #382 on August 7 2025) was **$8.91**.  

Rolling Stones albums we have:

- Hot Rocks, 1964‑1971 (Disc 1)  
- No Security  
- Voodoo Lounge  

Let me know if you’d like more details or want to place an order!
Assistant: **You

## 15. Test Case 2 — new session, no credentials provided

*"How much was my most recent purchase? What albums do you have by the Rolling Stones?"*

Expected: no customer_id/phone/email is present in the message, so `verify_customer` **interrupts**, asking the user to provide their Customer ID, Phone, or Email.


In [19]:
thread_id_2 = "notebook-test-case-2"
config_2 = {"configurable": {"thread_id": thread_id_2}}

result_3 = graph.invoke(
    {"messages": [HumanMessage(content=(
        "How much was my most recent purchase? What albums do you have by "
        "the Rolling Stones?"
    ))]},
    config=config_2,
)

assert "__interrupt__" in result_3, "expected an identity-verification interrupt"
question = result_3["__interrupt__"][0].value["question"]
print("Interrupt raised, as expected:")
print(question)

Interrupt raised, as expected:
I couldn't verify your identity. Please provide your Customer ID, Phone, or Email so I can verify your identity.


In [20]:
# Optionally, resume Test Case 2 with a valid identity to show the flow
# completing end-to-end after the interrupt.
result_3_resumed = graph.invoke(
    Command(resume="My customer id is 1"),
    config=config_2,
)
for m in result_3_resumed["messages"]:
    if isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
        print("Assistant:", m.content)

[supervisor] route=both (The customer asks for the amount of their most recent purchase (invoice information) and also requests a list of Rolling Stones albums (catalog lookup), requiring both specialists.)
Assistant: I’m sorry—I don’t have access to your purchase history, so I can’t tell you the amount of your most recent order.

Here are the Rolling Stones albums currently in our catalog:

- **Hot Rocks, 1964‑1971 (Disc 1)**
- **No Security**
- **Voodoo Lounge**

Let me know if you’d like more details about any of these albums or help finding tracks to purchase!
Assistant: Your most recent purchase was **$8.91** (Invoice #382 on August 7 2025).

**Rolling Stones albums we have:**

- *Hot Rocks, 1964‑1971 (Disc 1)*
- *No Security*
- *Voodoo Lounge*

Let me know if you’d like details or want to add any tracks from these albums to your cart!
Assistant: **Your most recent order**

- **Amount:** $8.91  
- **Invoice #:** 382  
- **Order date:** August 7 2025  

**Rolling Stones albums curr

## 16. Running outside the notebook

- CLI demo (runs both test cases non-interactively): `python -m src.run`
- Interactive CLI: `from src.run import run_conversation; run_conversation(thread_id="session-1")`
- Test suite: `pytest -q` (tool/verification tests need no API key; end-to-end tests are skipped automatically without `GROQ_API_KEY`)

See `README.md` for full setup instructions, architecture notes, HITL behavior, and known limitations.
